In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
import os
import sys
import json
from sklearn.metrics import root_mean_squared_error
from ase.io import read, write
import numpy as np
import time
import torch
from mace.calculators import MACECalculator
from pymatgen.io.vasp.outputs import Vasprun
from pathlib import Path
from sklearn.metrics import root_mean_squared_error
from ase.io import read, write
from sklearn.mixture import GaussianMixture
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize
from pymatgen.core import Structure
from dscribe.descriptors import SOAP
from ase.build import molecule
import math
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from IPython.display import clear_output

In [ ]:
n_50pts = MACECalculator("../data/ChClCA_Finetuning/Naive/Finetuned_Models/N-50pts/50pts.model", device='cpu')
ft5 = MACECalculator("../data/ChClCA_Finetuning/Periodic/Model:FT5/model/ft5.modell", device='cpu')

In [ ]:
class DFTInfo():
    def __init__(self, structure, forces, stress, energy, success=True):
        self.structure = structure
        self.forces = forces
        self.stress = stress
        self.energy = energy
        self.success = success
    def __str__(self):
        return self.structure, self.forces, self.energy
    def xyz_forces(self):
        if not self.success:
            return None
        arr = []
        for site, force in zip(self.structure.sites, self.forces):
            arr.append(np.concatenate([site.coords, force]))
        return np.array(arr)
    def from_json(cls, json_path):
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
            structure = Structure.from_dict(data["structure"])
            forces = np.array(data["forces"])
            stress = np.array(data["stress"])
            energy = data["energy"]
            return cls(structure, forces, stress, energy, success=True)
        except Exception as e:
            print(f"Error reading JSON {json_path}: {e}")
            return cls(None, None, None, None, success=False)
def get_DFT_from_directory(dir_path):
    """ Takes in directory, reads in JSON files, and outputs dictionary of (timestep, DFTInfo)
    """
    frames = {}
    for fname in sorted(os.listdir(dir_path)):
        timestep = int(fname.split(".")[1])
        frames[timestep] = DFTInfo.from_json(DFTInfo, os.path.join(dir_path, fname + '/vasp_info.json'))
    return frames